In [1]:
from pathlib import Path
import requests

url = "https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"

project_root = Path.cwd().parent
books_path = project_root / "input_docs" / "books"
books_path.mkdir(parents=True, exist_ok=True)

output_file = books_path / "books.csv"

response = requests.get(url, timeout=30)
response.raise_for_status()

output_file.write_bytes(response.content)

print(f"Downloaded to: {output_file}")

Downloaded to: /Users/mikifoster/Documents/Code_Projects/rags_to_agents/agent_experimental_builds/input_docs/books/books.csv


In [2]:
from pathlib import Path
import pandas as pd

books_path = Path("/Users/mikifoster/Documents/Code_Projects/rags_to_agents/agent_experimental_builds/input_docs/books")
csv_path = books_path / "books.csv"

df = pd.read_csv(csv_path)

print(df.columns)
df.head()

Index(['title', 'book_url', 'pdf_url'], dtype='str')


,title,book_url,pdf_url
0,Think Python 2e,https://greenteapress.com/wp/think-python-2e/,http://greenteapress.com/thinkpython2/thinkpyt...
1,Think DSP,https://greenteapress.com/wp/think-dsp/,http://greenteapress.com/thinkdsp/thinkdsp.pdf
2,Think Complexity 2e,https://greenteapress.com/wp/think-complexity/,http://greenteapress.com/complexity2/thinkcomp...
3,Think Java 2e,https://greenteapress.com/wp/think-java-2e/,http://greenteapress.com/thinkjava7/thinkjava2...
4,Physical Modeling in MATLAB,https://greenteapress.com/wp/physical-modeling...,https://github.com/AllenDowney/PhysicalModelin...


In [3]:
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
import requests
import re

BOOKS_PATH = Path("/Users/mikifoster/Documents/Code_Projects/rags_to_agents/agent_experimental_builds/input_docs/books")
CSV_PATH = books_path / "books.csv"
PDF_OUTPUT_PATH = Path("/Users/mikifoster/Documents/Code_Projects/rags_to_agents/agent_experimental_builds/output_docs/books_pdfs")


url_column = "pdf_url"

def clean_filename(value: str) -> str:
    value = value.strip()
    value = re.sub(r"[^a-zA-Z0-9._ -]", "", value)
    value = re.sub(r"\s+", "_", value)
    return value[:150]

def get_pdf_filename(pdf_url: str, fallback_name: str) -> str:
    parsed_url = urlparse(pdf_url)
    filename = Path(parsed_url.path).name or fallback_name
    if not filename.lower().endswith(".pdf"):
        filename = f"{Path(filename).stem}.pdf"
    return clean_filename(filename)

def download_pdf(pdf_url: str, output_file: Path) -> bool:
    if output_file.exists():
        print(f"Skipping existing file: {output_file.name}")
        return False

    response = requests.get(pdf_url, timeout=30)
    response.raise_for_status()
    output_file.write_bytes(response.content)
    print(f"Saved: {output_file.name}")
    return True


def download_pdfs_from_csv(
    csv_path: str | Path,
    output_path: str | Path,
    url_column: str,
    title_column: str | None = None,
) -> None:

    csv_path = Path(csv_path)
    output_path = Path(output_path)

    if not output_path.exists():

        raise FileNotFoundError(f"Output folder does not exist: {output_path}")

    df = pd.read_csv(csv_path)

    if url_column not in df.columns:

        raise ValueError(f"Missing expected URL column: {url_column}")

    for idx, record in enumerate(df.to_dict("records")):
        pdf_url = record.get(url_column)

        if pd.isna(pdf_url) or not str(pdf_url).strip():
            print(f"Skipping row {idx}: missing PDF URL")
            continue

        pdf_url = str(pdf_url).strip()

        if title_column and title_column in record and not pd.isna(record[title_column]):

            filename = clean_filename(str(record[title_column])) + ".pdf"

        else:

            filename = get_pdf_filename(pdf_url, fallback_name=f"book_{idx}.pdf")

        output_file = output_path / filename

        try:

            download_pdf(pdf_url, output_file)

        except requests.RequestException as e:

            print(f"Failed row {idx}: {pdf_url}")
            print(f"Error: {e}")

In [4]:
download_pdfs_from_csv(
    csv_path=CSV_PATH,
    output_path=PDF_OUTPUT_PATH,
    url_column="pdf_url",
    title_column="title",
)

Skipping existing file: Think_Python_2e.pdf
Skipping existing file: Think_DSP.pdf
Skipping existing file: Think_Complexity_2e.pdf
Skipping existing file: Think_Java_2e.pdf
Skipping existing file: Physical_Modeling_in_MATLAB.pdf
Skipping existing file: Think_OS.pdf
Skipping existing file: Think_C.pdf


In [5]:
from markitdown import MarkItDown

PDF_INPUT_PATH = Path(
    "/Users/mikifoster/Documents/Code_Projects/rags_to_agents/agent_experimental_builds/output_docs/books_pdfs"
)

MD_OUTPUT_PATH = Path(
    "/Users/mikifoster/Documents/Code_Projects/rags_to_agents/agent_experimental_builds/output_docs/books_md"
)

def convert_pdf_to_markdown(
    pdf_path: str | Path,
    md_output_path: str | Path,
    markitdown: MarkItDown,
    overwrite: bool = False,

) -> bool:

    pdf_path = Path(pdf_path)
    md_output_path = Path(md_output_path)
    output_file = md_output_path / f"{pdf_path.stem}.md"

    if output_file.exists() and not overwrite:

        print(f"Skipping existing file: {output_file.name}")

        return False

    result = markitdown.convert(str(pdf_path))
    output_file.write_text(result.text_content, encoding="utf-8")

    print(f"Saved: {output_file.name}")

    return True

def convert_all_pdfs_to_markdown(
    pdf_input_path: str | Path,
    md_output_path: str | Path,
    overwrite: bool = False,

) -> None:

    pdf_input_path = Path(pdf_input_path)
    md_output_path = Path(md_output_path)

    if not pdf_input_path.exists():

        raise FileNotFoundError(f"PDF input folder does not exist: {pdf_input_path}")

    if not md_output_path.exists():

        raise FileNotFoundError(f"Markdown output folder does not exist: {md_output_path}")

    pdf_files = sorted(pdf_input_path.glob("*.pdf"))

    if not pdf_files:

        raise FileNotFoundError(f"No PDF files found in: {pdf_input_path}")

    markitdown = MarkItDown()

    converted_count = 0
    skipped_count = 0
    failed_count = 0

    for pdf_path in pdf_files:

        try:

            converted = convert_pdf_to_markdown(
                pdf_path=pdf_path,
                md_output_path=md_output_path,
                markitdown=markitdown,
                overwrite=overwrite,
            )

            if converted:
                converted_count += 1

            else:
                skipped_count += 1

        except Exception as e:
            failed_count += 1
            print(f"Failed: {pdf_path.name}")
            print(f"Error: {e}")

    print()
    print(f"Converted: {converted_count}")
    print(f"Skipped: {skipped_count}")
    print(f"Failed: {failed_count}")

In [6]:
convert_all_pdfs_to_markdown(
    pdf_input_path=PDF_INPUT_PATH,
    md_output_path=MD_OUTPUT_PATH,
    overwrite=False,
)

Skipping existing file: Physical_Modeling_in_MATLAB.md
Skipping existing file: Think_C.md
Skipping existing file: Think_Complexity_2e.md
Skipping existing file: Think_DSP.md
Skipping existing file: Think_Java_2e.md
Skipping existing file: Think_OS.md
Skipping existing file: Think_Python_2e.md

Converted: 0
Skipped: 7
Failed: 0


First, prepare your documents: 

Read each markdown file from your books_text/ directory
Split the content into lines
Remove empty lines and lines that contain only whitespace
Turn each book into a dictionary with source (filename) and a content (list of non-empty lines)

After that, chunk it.
Use the gitsource package which provides the chunk_documents function.

In [7]:
from pathlib import Path
from minsearch import Index
import re

BOOKS_TEXT_PATH = Path("/Users/mikifoster/Documents/Code_Projects/rags_to_agents/agent_experimental_builds/output_docs/books_md")


def read_markdown_file(markdown_path: str | Path) -> dict:
    markdown_path = Path(markdown_path)
    lines = markdown_path.read_text(encoding="utf-8").splitlines()
    non_empty_lines = [line.strip() for line in lines if line.strip()]
    return {
        "filename": markdown_path.name,
        "source": markdown_path.name,
        "content": non_empty_lines,
    }

def load_books_from_markdown(books_text_path: str | Path) -> list[dict]:
    books_text_path = Path(books_text_path)
    markdown_files = sorted(books_text_path.glob("*.md"))
    return [read_markdown_file(markdown_path) for markdown_path in markdown_files]

def sliding_window(items: list[str], size: int = 100, step: int = 50) -> list[dict]:
    chunks = []
    start = 0
    item_count = len(items)

    while start < item_count:
        end = start + size
        chunk_items = items[start:end]
        chunks.append({
            "start": start,
            "content": chunk_items,
        })

        if end >= item_count:
            break

        start += step

    return chunks

def build_document_chunks(parsed_files: list[dict], size: int = 100, step: int = 50) -> list[dict]:
    document_chunks = []

    for file in parsed_files:
        if not file.get("content"):
            continue

        metadata = file.copy()
        content = metadata.pop("content")

        chunks = sliding_window(content, size=size, step=step)

        for i, chunk in enumerate(chunks):
            chunk.update(metadata)
            chunk["chunk_id"] = i
            chunk["content"] = "\n".join(chunk["content"])
            document_chunks.append(chunk)

    return document_chunks

parsed_files = load_books_from_markdown(BOOKS_TEXT_PATH)

document_chunks = build_document_chunks(
    parsed_files,
    size=100,
    step=50,
)

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(document_chunks)

print(f"Loaded {len(parsed_files)} books")
print(f"Created {len(document_chunks)} chunks")
print(f"Indexed {len(document_chunks)} chunks")

Loaded 7 books
Created 1009 chunks
Indexed 1009 chunks


In [8]:
think_python_chunks = [
    chunk
    for chunk in document_chunks
    if chunk["filename"] == "Think_Python_2e.md"
]

print(len(think_python_chunks))


214


In [9]:
print(len(document_chunks))

1009


In [10]:
print(f"Number of parsed files: {len(parsed_files)}")
print(f"Number of document chunks: {len(document_chunks)}")

for file in parsed_files:
    print(file["filename"], len(file["content"]))

Number of parsed files: 7
Number of document chunks: 1009
Physical_Modeling_in_MATLAB.md 5978
Think_C.md 5371
Think_Complexity_2e.md 7216
Think_DSP.md 5524
Think_Java_2e.md 12373
Think_OS.md 3444
Think_Python_2e.md 10729


In [11]:
def group_chunks_by_filename(document_chunks: list[dict]) -> dict[str, list[dict]]:
    chunks_by_filename = {}

    for chunk in document_chunks:
        filename = chunk["filename"]
        chunks_by_filename.setdefault(filename, []).append(chunk)

    return chunks_by_filename


chunks_by_filename = group_chunks_by_filename(document_chunks)

In [12]:
for filename, book_chunks in sorted(chunks_by_filename.items()):
    print(filename, len(book_chunks))

Physical_Modeling_in_MATLAB.md 119
Think_C.md 107
Think_Complexity_2e.md 144
Think_DSP.md 110
Think_Java_2e.md 247
Think_OS.md 68
Think_Python_2e.md 214


In [13]:
from gitsource import chunk_documents

gitsource_chunks = chunk_documents(
    parsed_files,
    size=100,
    step=50,
)

print(len(gitsource_chunks))

1009


In [26]:
results = chunk_index.search("python function definition", num_results=5)
results[0]['source']


'Think_Python_2e.md'

In [27]:
import json
from openai import OpenAI


class IndexedPromptResponseRunner:
    MODEL_PRICES = {
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "gpt-5-nano": {"input": 0.075, "output": 0.30},
        "gpt-5-mini": {"input": 0.25, "output": 2.00},
        "gpt-5.2": {"input": 1.75, "output": 14.00},
        "gpt-5.2-pro": {"input": 21.00, "output": 168.00},
        "claude-haiku-4.5": {"input": 1.00, "output": 5.00},
        "claude-sonnet-4.6": {"input": 3.00, "output": 15.00},
        "claude-opus-4.7": {"input": 5.00, "output": 25.00},
        "claude-opus-4.6": {"input": 5.00, "output": 25.00},
        "claude-opus-4.5": {"input": 5.00, "output": 25.00},
        "claude-sonnet-4.5": {"input": 3.00, "output": 15.00},
        "claude-3.5-haiku": {"input": 0.80, "output": 4.00},
        "claude-3-haiku": {"input": 0.25, "output": 1.25},
    }

    DEFAULT_INSTRUCTIONS = """
You're a course assistant, your task is to answer the PROMPT from the
course students using the provided CONTEXT.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
""".strip()

    DEFAULT_PROMPT_TEMPLATE = """
<PROMPT>
{prompt}
</PROMPT>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

    def __init__(
        self,
        index,
        client=None,
        default_model="gpt-4o-mini",
        default_instructions=None,
        prompt_template=None,
        default_num_results=5,
    ):
        self.index = index
        self.client = client or OpenAI()
        self.default_model = default_model
        self.default_instructions = default_instructions or self.DEFAULT_INSTRUCTIONS
        self.prompt_template = prompt_template or self.DEFAULT_PROMPT_TEMPLATE
        self.default_num_results = default_num_results

    def search_index(self, prompt: str, num_results: int | None = None) -> list[dict]:
        selected_num_results = self.default_num_results if num_results is None else num_results
        return self.index.search(
            query=prompt,
            num_results=selected_num_results,
        )

    def build_prompt(self, prompt: str, search_results: list[dict]) -> str:
        context = json.dumps(search_results, indent=2)
        return self.prompt_template.format(
            prompt=prompt,
            context=context,
        ).strip()

    def call_model(
        self,
        prompt: str,
        instructions: str,
        model: str,
        response_model=None,
    ):
        messages = [
            {"role": "system", "content": instructions},
            {"role": "user", "content": prompt},
        ]

        if response_model is None:
            return self.client.responses.create(
                model=model,
                input=messages,
            )

        return self.client.responses.parse(
            model=model,
            input=messages,
            text_format=response_model,
        )

    def calculate_cost(
        self,
        model_name: str,
        input_tokens: int,
        output_tokens: int,
    ) -> float:
        prices = self.MODEL_PRICES[model_name.lower()]
        input_cost = (input_tokens / 1_000_000) * prices["input"]
        output_cost = (output_tokens / 1_000_000) * prices["output"]
        return input_cost + output_cost

    def extract_answer(self, response, response_model=None) -> str:
        if response_model is None:
            return response.output_text

        return response.output_parsed.answer

    def run(
        self,
        prompt: str,
        instructions: str | None = None,
        model: str | None = None,
        num_results: int | None = None,
        response_model=None,
    ) -> dict:
        original_prompt = prompt
        selected_model = model or self.default_model
        selected_instructions = instructions or self.default_instructions
        selected_num_results = self.default_num_results if num_results is None else num_results

        search_results = self.search_index(
            prompt=original_prompt,
            num_results=selected_num_results,
        )

        final_prompt = self.build_prompt(
            prompt=original_prompt,
            search_results=search_results,
        )

        response = self.call_model(
            prompt=final_prompt,
            instructions=selected_instructions,
            model=selected_model,
            response_model=response_model,
        )

        usage = response.usage

        result = {
            "answer": self.extract_answer(response, response_model=response_model),
            "prompt": original_prompt,
            "final_prompt": final_prompt,
            "instructions": selected_instructions,
            "search_results": search_results,
            "input_tokens": usage.input_tokens,
            "output_tokens": usage.output_tokens,
            "total_tokens": usage.total_tokens,
            "cost": self.calculate_cost(
                model_name=selected_model,
                input_tokens=usage.input_tokens,
                output_tokens=usage.output_tokens,
            ),
            "model": selected_model,
            "num_results": selected_num_results,
        }

        if response_model is not None:
            result["parsed_response"] = response.output_parsed

        return result

In [29]:
book_prompt_runner = IndexedPromptResponseRunner(
    index=chunk_index,
    default_model="gpt-4o-mini",
    default_num_results=5,
)

rag_response = book_prompt_runner.run(
    prompt="python function definition"
)

print(rag_response["answer"])
print(f"input tokens: {rag_response["input_tokens"]}")
print(f"output tokens: {rag_response["output_tokens"]}")
print(f"total tokens: {rag_response["total_tokens"]}")
print(f"Cost: ${rag_response['cost']:.6f}")

In Python, a function is defined using the `def` keyword followed by the function name and parentheses. Inside the parentheses, you can specify any parameters the function will accept. The body of the function contains the statements that define its behavior.

Here is a basic structure for a function definition in Python:

```python
def function_name(parameters):
    # Function body
    statements
    return value  # Optional
```

For example, the "Hello, World!" program mentioned earlier can be represented as a function:

```python
def hello_world():
    print('Hello, World!')
```

To call this function, you would simply use:

```python
hello_world()
```

This will execute the function and print "Hello, World!" on the screen.
input tokens: 7447
output tokens: 155
total tokens: 7602
Cost: $0.001210


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class RAGResponse(BaseModel):
    answer: str = Field(description="The most true answer to the user's prompt in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the retrieved context")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation of the confidence level")
    answer_type: Literal[
        "how-to",
        "explanation",
        "troubleshooting",
        "comparison",
        "reference",
        "not-found",
    ] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(default_factory=list, description="Suggested follow-up questions")

In [31]:
unstructured_response = book_prompt_runner.run(
    prompt="python function definition"
)

print(unstructured_response["answer"])
print(unstructured_response["input_tokens"])
print(unstructured_response["output_tokens"])
print(unstructured_response["total_tokens"])

In Python, a function is defined using the `def` keyword followed by the function name and parentheses. The syntax is as follows:

```python
def function_name(parameters):
    # function body
    return value  # optional
```

The first line is called the function signature, which specifies the name of the function and any parameters it takes. The body contains the statements that define what the function does. 

For example, a simple function could look like this:

```python
def add(a, b):
    return a + b
```

When you call this function with values, it will execute and return the result of the addition.
7447
132
7579


In [32]:
structured_response = book_prompt_runner.run(
    prompt="python function definition",
    response_model=RAGResponse,
)

print(structured_response["parsed_response"])
print(structured_response["answer"])
print(structured_response["input_tokens"])
print(structured_response["output_tokens"])
print(structured_response["total_tokens"])

answer="In Python, a function definition allows you to create a reusable block of code that performs a specific task. The basic syntax for defining a function is as follows:\n\n```python\ndef function_name(parameters):\n    # function body\n    return result\n```\n\n### Components of a Function Definition:\n1. **`def` keyword:** Indicates that you are defining a function.\n2. **Function name:** This is how you will refer to the function later on. It should be descriptive and follow standard naming conventions (e.g., `my_function`).\n3. **Parameters:** These are inputs that can be passed to the function. They are optional - if your function doesn't need any inputs, you can leave the parentheses empty.\n4. **Function body:** This is the indented block of code that defines what the function does.\n5. **Return statement:** This is optional and is used to return a value from the function back to the caller. If you omit this, the function will return `None` by default.\n\n### Example:\nHere’

In [33]:
structured_response["output_tokens"] - unstructured_response["output_tokens"]

281